<a href="https://colab.research.google.com/github/dwiiittt/RAG_Chatbot/blob/main/movie_rag_chatbot_(2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎬 Movie RAG Chatbot — IMDB Dataset

Chatbot berbasis **Retrieval-Augmented Generation (RAG)** menggunakan dataset film IMDB.

### Fitur utama:
- ✅ **Gratis** — menggunakan **Groq API** (free tier, tanpa kartu kredit)
- ✅ **Multi-turn conversation** — percakapan nyambung antar giliran
- ✅ **Context-aware retrieval** — query FAISS diperkaya konteks percakapan sebelumnya
- ✅ **Graceful fallback** — jika pertanyaan di luar topik film, bot mengatakan tidak tahu + saran situs
- ✅ **Session state** — setiap sesi pengguna Gradio punya history sendiri

### Cara mendapatkan Groq API Key (GRATIS):
1. Buka https://console.groq.com
2. Daftar / login (bisa pakai Google)
3. Klik **API Keys** → **Create API Key**
4. Copy key-nya, paste saat diminta di bawah

> **Rate limit Groq (gratis):** ~30 request/menit, 14.400 request/hari — lebih dari cukup untuk chatbot.

## 📦 1. Instalasi Library

In [ ]:
!pip install groq sentence-transformers faiss-cpu gradio pandas numpy tqdm --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 42.2 MB/s eta 0:00:00


## 🔑 2. Konfigurasi Groq API Key

In [ ]:
import os
import getpass

# Dapatkan API key gratis di: https://console.groq.com
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Masukkan Groq API Key Anda (gratis): ")

# Verifikasi koneksi
from groq import Groq
client = Groq(api_key=os.environ["GROQ_API_KEY"])
test = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": "Balas hanya: OK"}],
    max_tokens=5,
)
print(f"✅ Groq API terhubung! Response: {test.choices[0].message.content}")

Masukkan Groq API Key Anda (gratis): ··········
✅ Groq API terhubung! Response: OK


## 📂 3. Load & Preprocessing Dataset

In [ ]:
import pandas as pd
import numpy as np

CSV_PATH = "imdb_movies.csv"  # Sesuaikan path jika berbeda

df = pd.read_csv(CSV_PATH)
print(f"📊 Total baris awal: {len(df)}")
print(f"📋 Kolom: {list(df.columns)}")
df.head(3)

📊 Total baris awal: 10178
📋 Kolom: ['names', 'date_x', 'score', 'genre', 'overview', 'crew', 'orig_title', 'status', 'orig_lang', 'budget_x', 'revenue', 'country']


,names,date_x,score,genre,overview,crew,orig_title,status,orig_lang,budget_x,revenue,country
0,Creed III,03/02/2023,73.0,"Drama, Action","After dominating the boxing world, Adonis Cree...","Michael B. Jordan, Adonis Creed, Tessa Thompso...",Creed III,Released,English,75000000.0,2.716167e+08,AU
1,Avatar: The Way of Water,12/15/2022,78.0,"Science Fiction, Adventure, Action",Set more than a decade after the events of the...,"Sam Worthington, Jake Sully, Zoe Saldaña, Neyt...",Avatar: The Way of Water,Released,English,460000000.0,2.316795e+09,AU
2,The Super Mario Bros. Movie,04/05/2023,76.0,"Animation, Adventure, Family, Fantasy, Comedy","While working underground to fix a water main,...","Chris Pratt, Mario (voice), Anya Taylor-Joy, P...",The Super Mario Bros. Movie,Released,English,100000000.0,7.244590e+08,AU


In [ ]:
# ─── Cleaning ──────────────────────────────────────────────────────────────────
str_cols = ['names', 'date_x', 'genre', 'overview', 'crew',
            'orig_title', 'status', 'orig_lang', 'country']
for col in str_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()

df['genre']    = df['genre'].fillna('Unknown')
df['crew']     = df['crew'].fillna('Unknown')
df['overview'] = df['overview'].fillna('No overview available.')

df = df.drop_duplicates(subset='names').reset_index(drop=True)
df = df[df['overview'].str.len() > 20].reset_index(drop=True)

print(f"✅ Total baris setelah cleaning: {len(df)}")

✅ Total baris setelah cleaning: 9652


In [ ]:
# ─── Buat Dokumen Teks per Film ────────────────────────────────────────────────

def format_money(val):
    try:
        v = float(val)
        if v >= 1e9:   return f"${v/1e9:.2f} miliar"
        elif v >= 1e6: return f"${v/1e6:.1f} juta"
        elif v > 0:    return f"${v:,.0f}"
        else:          return "tidak tersedia"
    except:
        return "tidak tersedia"


def build_doc(row):
    title      = row.get('names', 'Unknown')
    orig       = row.get('orig_title', title)
    date       = row.get('date_x', 'Unknown')
    genre      = row.get('genre', 'Unknown')
    score      = row.get('score', 'N/A')
    status     = row.get('status', 'Unknown')
    lang       = row.get('orig_lang', 'Unknown')
    country    = row.get('country', 'Unknown')
    overview   = row.get('overview', '')
    crew       = row.get('crew', 'Unknown')
    budget     = format_money(row.get('budget_x', 0))
    revenue    = format_money(row.get('revenue', 0))
    crew_short = ', '.join(crew.split(',')[:6]) if crew != 'Unknown' else 'Unknown'

    return (
        f"Judul: {title}\n"
        f"Judul Asli: {orig}\n"
        f"Tanggal Rilis: {date}\n"
        f"Genre: {genre}\n"
        f"Skor IMDB: {score}/100\n"
        f"Status: {status}\n"
        f"Bahasa: {lang} | Negara: {country}\n"
        f"Budget: {budget} | Pendapatan: {revenue}\n"
        f"Pemeran & Kru: {crew_short}\n"
        f"Sinopsis: {overview}"
    )


df['document'] = df.apply(build_doc, axis=1)
print("📄 Contoh dokumen film:")
print(df['document'].iloc[0])
print(f"\n✅ Total dokumen dibuat: {len(df)}")

📄 Contoh dokumen film:
Judul: Creed III
Judul Asli: Creed III
Tanggal Rilis: 03/02/2023
Genre: Drama, Action
Skor IMDB: 73.0/100
Status: Released
Bahasa: English | Negara: AU
Budget: $75.0 juta | Pendapatan: $271.6 juta
Pemeran & Kru: Michael B. Jordan,  Adonis Creed,  Tessa Thompson,  Bianca Taylor,  Jonathan Majors,  Damien Anderson
Sinopsis: After dominating the boxing world, Adonis Creed has been thriving in both his career and family life. When a childhood friend and former boxing prodigy, Damien Anderson, resurfaces after serving a long sentence in prison, he is eager to prove that he deserves his shot in the ring. The face-off between former friends is more than just a fight. To settle the score, Adonis must put his future on the line to battle Damien — a fighter who has nothing to lose.

✅ Total dokumen dibuat: 9652


## 🧠 4. Embedding & Vector Store (FAISS)

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import pickle

EMBEDDING_MODEL = "all-MiniLM-L6-v2"   # 384 dimensi, ringan & cepat, GRATIS (lokal)
INDEX_PATH      = "movie_faiss.index"
DOCS_PATH       = "movie_docs.pkl"

print(f"⏳ Memuat model embedding: {EMBEDDING_MODEL} ...")
embedder = SentenceTransformer(EMBEDDING_MODEL)
print("✅ Model embedding siap.")

⏳ Memuat model embedding: all-MiniLM-L6-v2 ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Model embedding siap.


In [ ]:
# ─── Buat Embedding & FAISS Index ─────────────────────────────────────────────
# Proses ini ~3-5 menit untuk 10K film (hanya perlu dijalankan sekali)

documents = df['document'].tolist()

print(f"⏳ Membuat embedding untuk {len(documents):,} film ...")
embeddings = embedder.encode(
    documents,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)
print(f"✅ Embedding shape: {embeddings.shape}")

dim   = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings)
print(f"✅ FAISS index — {index.ntotal:,} vektor tersimpan.")

faiss.write_index(index, INDEX_PATH)
with open(DOCS_PATH, 'wb') as f:
    pickle.dump(documents, f)
print(f"💾 Disimpan: '{INDEX_PATH}' dan '{DOCS_PATH}'")

⏳ Membuat embedding untuk 9,652 film ...


Batches:   0%|          | 0/151 [00:00<?, ?it/s]

✅ Embedding shape: (9652, 384)
✅ FAISS index — 9,652 vektor tersimpan.
💾 Disimpan: 'movie_faiss.index' dan 'movie_docs.pkl'


## 🔄 5. RAG Pipeline dengan Multi-Turn Memory

### LLM: Groq (gratis)
Model yang digunakan: **`llama-3.3-70b-versatile`** — model LLaMA 70B dari Meta, dijalankan di chip Groq yang sangat cepat.

| Strategi | Penjelasan |
|----------|------------|
| **Contextual Query Expansion** | Query pendek ("Siapa pemerannya?") diperkaya kata kunci dari history sebelum dikirim ke FAISS |
| **Full History Injection** | Seluruh riwayat turn dikirim ke Groq API setiap request |
| **`gr.State`** | Setiap sesi Gradio punya history list sendiri, tidak tercampur |

In [ ]:
from groq import Groq
from sentence_transformers import SentenceTransformer
import faiss
import pickle
import os

# ─── Load artefak (jalankan cell ini jika kernel direstart) ───────────────────
INDEX_PATH = "movie_faiss.index"
DOCS_PATH  = "movie_docs.pkl"

index = faiss.read_index(INDEX_PATH)
with open(DOCS_PATH, 'rb') as f:
    documents = pickle.load(f)

embedder = SentenceTransformer("all-MiniLM-L6-v2")
client   = Groq(api_key=os.environ["GROQ_API_KEY"])

print(f"✅ FAISS index: {index.ntotal:,} vektor | Dokumen: {len(documents):,}")
print(f"✅ Groq client siap.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ FAISS index: 9,652 vektor | Dokumen: 9,652
✅ Groq client siap.


In [ ]:
# ─── Konfigurasi ───────────────────────────────────────────────────────────────

# Model Groq yang tersedia gratis:
# - "llama-3.3-70b-versatile"   ← RECOMMENDED: paling pintar
# - "llama-3.1-8b-instant"      ← paling cepat (tapi kurang akurat)
# - "gemma2-9b-it"              ← alternatif Google Gemma
# - "mixtral-8x7b-32768"        ← context window besar
LLM_MODEL = "llama-3.3-70b-versatile"

TOP_K                = 5
SIMILARITY_THRESHOLD = 0.25
MAX_TOKENS           = 1024
MAX_HISTORY_TURNS    = 6    # Batasi history → hemat token

RECOMMENDED_SITES = [
    "🌐 **IMDb** (imdb.com) — database film & TV terlengkap",
    "🍅 **Rotten Tomatoes** (rottentomatoes.com) — agregasi ulasan kritikus",
    "🎬 **TMDb** (themoviedb.org) — database film komunitas terbuka",
    "📺 **Letterboxd** (letterboxd.com) — komunitas pecinta film",
    "🎥 **Metacritic** (metacritic.com) — skor agregasi ulasan",
]

# Groq menggunakan format OpenAI-compatible:
# system prompt dimasukkan sebagai message pertama dengan role "system"
SYSTEM_PROMPT = """Anda adalah asisten chatbot khusus film yang berpengetahuan luas.
Anda HANYA menjawab pertanyaan yang berkaitan dengan film, sinema, atau konten audiovisual.

ATURAN PENTING:
1. Gunakan HANYA informasi dari [KONTEKS FILM] yang diberikan untuk menjawab pertanyaan.
2. Anda boleh merujuk ke jawaban Anda sebelumnya dalam percakapan ini untuk menjawab pertanyaan lanjutan.
3. Jika pertanyaan merujuk ke film yang sudah dibahas (contoh: "siapa pemerannya?", "berapa budgetnya?"),
   jawab berdasarkan konteks percakapan sebelumnya.
4. Jika informasi tidak ada dalam konteks maupun history, katakan tidak tahu dengan jujur.
5. JANGAN mengarang fakta atau informasi yang tidak ada dalam konteks.
6. Jika pertanyaan tidak berkaitan dengan film sama sekali, tolak dengan sopan.
7. Jawab dalam Bahasa Indonesia kecuali pengguna menggunakan bahasa lain.
8. Jawab dengan ramah, informatif, dan terstruktur. Gunakan emoji sesekali."""

print(f"✅ Konfigurasi: model={LLM_MODEL}, top_k={TOP_K}, threshold={SIMILARITY_THRESHOLD}")

✅ Konfigurasi: model=llama-3.3-70b-versatile, top_k=5, threshold=0.25


In [ ]:
# ─── Helper Functions ──────────────────────────────────────────────────────────

MOVIE_KEYWORDS = [
    'film', 'movie', 'sinema', 'bioskop', 'genre', 'sutradara', 'aktor',
    'aktris', 'pemeran', 'cerita', 'plot', 'sinopsis', 'rating', 'rilis',
    'tayang', 'produksi', 'budget', 'pendapatan', 'review', 'rekomendasi',
    'series', 'serial', 'episode', 'dokumenter', 'animasi', 'blockbuster',
    'oscar', 'award', 'penghargaan', 'trailer', 'streaming', 'netflix',
    'watch', 'cast', 'director', 'actor', 'actress', 'horror', 'comedy',
    'action', 'thriller', 'romance', 'sci-fi', 'fantasy', 'drama',
    'imdb', 'score', 'sequel', 'prequel', 'remake', 'karakter', 'tokoh',
    'siapa', 'berapa', 'kapan', 'dimana',
]

def is_movie_related(query: str) -> bool:
    return any(kw in query.lower() for kw in MOVIE_KEYWORDS)


def expand_query_with_context(query: str, history: list, n_last: int = 2) -> str:
    """
    Perkaya query dengan kata kunci dari history untuk retrieval FAISS yang lebih akurat.
    Contoh: "Siapa pemerannya?" + history[Avatar] → "Siapa pemerannya? Avatar Way Water"
    """
    if not history:
        return query
    recent = history[-(n_last * 2):]
    topic_words = []
    for msg in recent:
        if msg["role"] == "user":
            words = [w for w in msg["content"].split() if len(w) > 3]
            topic_words.extend(words[:5])
    if not topic_words:
        return query
    context_hint = ' '.join(dict.fromkeys(topic_words))
    return f"{query} {context_hint}"


def retrieve(query: str, top_k: int = TOP_K) -> list:
    q_vec = embedder.encode(
        [query], normalize_embeddings=True, convert_to_numpy=True
    )
    scores, indices = index.search(q_vec, top_k)
    return [
        {"doc": documents[idx], "score": float(score)}
        for score, idx in zip(scores[0], indices[0])
        if idx >= 0 and score >= SIMILARITY_THRESHOLD
    ]


print("✅ Helper functions siap.")

✅ Helper functions siap.


In [ ]:
# ─── Fungsi Utama RAG Chat (Groq) ─────────────────────────────────────────────

def rag_chat(query: str, history: list) -> tuple[str, list]:
    """
    Multi-turn RAG chatbot menggunakan Groq API.

    Args:
        query   : Pertanyaan user saat ini
        history : Riwayat percakapan format OpenAI-compatible:
                  [{"role": "user"|"assistant", "content": "..."}]
    Returns:
        (jawaban_bot, history_terbaru)
    """
    # ── 1. Expand query dengan konteks history ────────────────────────────────
    expanded_query = expand_query_with_context(query, history)

    # ── 2. Retrieve dokumen relevan ───────────────────────────────────────────
    results = retrieve(expanded_query)

    # ── 3. Tangani kasus tidak ada dokumen relevan ────────────────────────────
    if not results:
        is_followup = (len(history) > 0 and len(query.split()) <= 10)

        if is_followup:
            # Pertanyaan lanjutan singkat — biarkan LLM jawab dari history
            context_block = "[Tidak ada konteks film baru — jawab berdasarkan percakapan sebelumnya jika relevan.]"
        elif not is_movie_related(query):
            fallback = (
                "Maaf, saya hanya bisa menjawab pertanyaan seputar film. "
                "Pertanyaan Anda sepertinya di luar topik tersebut. 😊\n\n"
                "Untuk referensi film, silakan kunjungi:\n\n"
                + "\n".join(RECOMMENDED_SITES)
            )
            history.append({"role": "user",      "content": query})
            history.append({"role": "assistant", "content": fallback})
            return fallback, history
        else:
            fallback = (
                "Saya tidak menemukan informasi tentang film tersebut dalam database saya. "
                "Database saya mencakup sekitar 10.000 judul film.\n\n"
                "Coba cari di:\n\n"
                + "\n".join(RECOMMENDED_SITES)
            )
            history.append({"role": "user",      "content": query})
            history.append({"role": "assistant", "content": fallback})
            return fallback, history
    else:
        context_parts = [f"[Film {i+1}]\n{r['doc']}" for i, r in enumerate(results)]
        context_block = "\n\n".join(context_parts)

    # ── 4. Bangun messages untuk Groq API (format OpenAI-compatible) ──────────
    # Groq tidak punya parameter 'system' terpisah — masukkan sebagai message pertama
    trimmed_history = history[-(MAX_HISTORY_TURNS * 2):]

    user_msg_with_context = (
        f"[KONTEKS FILM DARI DATABASE]\n"
        f"{'='*60}\n"
        f"{context_block}\n"
        f"{'='*60}\n\n"
        f"{query}"
    )

    messages_to_send = (
        [{"role": "system", "content": SYSTEM_PROMPT}]
        + list(trimmed_history)
        + [{"role": "user", "content": user_msg_with_context}]
    )

    # ── 5. Panggil Groq API ───────────────────────────────────────────────────
    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=messages_to_send,
        max_tokens=MAX_TOKENS,
        temperature=0.3,   # Lebih rendah = lebih faktual, kurang halusinasi
    )
    answer = response.choices[0].message.content

    # ── 6. Simpan query ASLI (tanpa konteks RAG) ke history ──────────────────
    history.append({"role": "user",      "content": query})
    history.append({"role": "assistant", "content": answer})

    return answer, history


print("✅ Fungsi rag_chat() dengan Groq API siap.")

✅ Fungsi rag_chat() dengan Groq API siap.


## 🧪 6. Testing Multi-Turn Conversation

In [ ]:
# ─── Test 1: Multi-turn — pertanyaan lanjutan ─────────────────────────────────
print("=" * 65)
print("TEST 1: Multi-turn — follow-up questions")
print("=" * 65)

history = []
turns = [
    "Ceritakan tentang film Avatar: The Way of Water",
    "Siapa saja pemeran utamanya?",          # follow-up → harus nyambung
    "Berapa pendapatan box office-nya?",     # follow-up → harus nyambung
    "Rekomendasikan film serupa genre-nya",  # pivot ke film lain
]

for i, q in enumerate(turns, 1):
    print(f"\n🧑 Turn {i}: {q}")
    answer, history = rag_chat(q, history)
    preview = answer[:300] + ('...' if len(answer) > 300 else '')
    print(f"🤖 Bot: {preview}")
    print(f"   [History: {len(history)} entries]")

print("\n✅ Test 1 selesai!")

TEST 1: Multi-turn — follow-up questions

🧑 Turn 1: Ceritakan tentang film Avatar: The Way of Water
🤖 Bot: 🌊 Film "Avatar: The Way of Water" adalah sebuah film Science Fiction, Adventure, dan Action yang dirilis pada tanggal 15 Desember 2022. Film ini merupakan sekuel dari film Avatar sebelumnya dan berlatar lebih dari satu dekade setelah peristiwa film pertama.

🌴 Cerita film ini berfokus pada keluarga ...
   [History: 2 entries]

🧑 Turn 2: Siapa saja pemeran utamanya?
🤖 Bot: 🤔 Berdasarkan konteks film yang diberikan, pemeran utama dari beberapa film adalah:

* Film 1: Tri Âm: Người Giữ Thời Gian - tidak ada informasi tentang pemeran utama
* Film 2: Avatar: Scene Deconstruction - tidak ada informasi tentang pemeran utama
* Film 3: Samsara - Ni Made Megahadi Pratiwi, Puti...
   [History: 4 entries]

🧑 Turn 3: Berapa pendapatan box office-nya?
🤖 Bot: 🎬 Berdasarkan konteks film yang diberikan, pendapatan box office dari beberapa film adalah:

* Tri Âm: Người Giữ Thời Gian: $175.3 juta
* 

In [ ]:
# ─── Test 2: Off-topic fallback ───────────────────────────────────────────────
print("=" * 65)
print("TEST 2: Off-topic fallback")
print("=" * 65)

history = []
for q in ["Apa resep nasi goreng?", "Berapa kurs dolar hari ini?"]:
    print(f"\n🧑 User: {q}")
    answer, history = rag_chat(q, history)
    print(f"🤖 Bot: {answer[:300]}")

print("\n✅ Test 2 selesai!")

TEST 2: Off-topic fallback

🧑 User: Apa resep nasi goreng?
🤖 Bot: Maaf, saya tidak bisa membantu dengan resep nasi goreng karena itu tidak terkait dengan film. Saya hanya bisa membantu dengan pertanyaan tentang film, sinema, atau konten audiovisual. Jika Anda memiliki pertanyaan tentang film, saya akan dengan senang hati membantu! 🎥

🧑 User: Berapa kurs dolar hari ini?
🤖 Bot: Maaf, saya tidak bisa membantu dengan informasi tentang kurs dolar hari ini karena itu tidak terkait dengan film. Saya hanya bisa membantu dengan pertanyaan tentang film, sinema, atau konten audiovisual yang ada dalam konteks yang diberikan. Jika Anda memiliki pertanyaan tentang film, saya akan deng

✅ Test 2 selesai!


## 🚀 7. Deploy ke Gradio

In [ ]:
import gradio as gr

def gradio_chat(user_message: str, chat_display: list, session_history: list):
    """
    Wrapper Gradio → rag_chat().
    - chat_display   : [[user_msg, bot_msg], ...] untuk gr.Chatbot
    - session_history: [{role, content}, ...] state Groq per sesi
    """
    if not user_message.strip():
        return "", chat_display, session_history

    answer, updated_history = rag_chat(user_message, list(session_history))
    chat_display = chat_display + [[user_message, answer]]
    return "", chat_display, updated_history


def reset_session():
    return [], []


EXAMPLES = [
    "Rekomendasi film horror terbaik",
    "Ceritakan tentang film Avatar: The Way of Water",
    "Film dengan pendapatan box office tertinggi?",
    "Siapa pemeran utama The Super Mario Bros. Movie?",
    "Rekomendasikan film animasi untuk anak-anak",
    "Film action dengan budget terbesar?",
    "Apa resep nasi goreng?",
]

# ─── Build UI ─────────────────────────────────────────────────────────────────
with gr.Blocks(
    title="🎬 Movie RAG Chatbot",
    theme=gr.themes.Soft(
        primary_hue="indigo",
        secondary_hue="purple",
        font=[gr.themes.GoogleFont("Inter"), "sans-serif"],
    ),
    css="""
    #header { text-align: center; padding: 16px 0 8px; }
    #header h1 { font-size: 2em; margin-bottom: 4px; }
    .tip { font-size: 0.82em; color: #888; }
    .send-row { align-items: flex-end; }
    """,
) as demo:

    # gr.State: history per sesi — tidak tercampur antar user/tab
    session_history = gr.State([])

    # Header
    with gr.Row(elem_id="header"):
        gr.Markdown(
            """
            # 🎬 Movie RAG Chatbot
            **Chatbot AI khusus film — RAG + Multi-turn + Groq (Gratis!)**

            Database: **~10.000 film IMDB** · Tanya soal sinopsis, genre, pemeran, rating, budget, dan lainnya!
            """
        )

    # Info cards
    with gr.Row():
        gr.Markdown(
            """### ✅ Yang bisa saya bantu:
- 🔍 Cari & bandingkan film
- 📖 Sinopsis dan detail lengkap
- ⭐ Rating & skor IMDB
- 💰 Budget & pendapatan box office
- 🎭 Rekomendasi berdasarkan selera
- 💬 **Percakapan multi-giliran** yang nyambung"""
        )
        gr.Markdown(
            """### ❌ Di luar jangkauan:
- 🚫 Topik non-film
- 🚫 Film tidak ada dalam database
- 🚫 Informasi real-time / terbaru

<p class='tip'>Untuk info terkini:<br>
imdb.com · rottentomatoes.com · themoviedb.org</p>"""
        )

    gr.Markdown("---")

    # Chat display
    chatbot_display = gr.Chatbot(
        label="Percakapan",
        height=460,
        show_copy_button=True,
        bubble_full_width=False,
        placeholder="<center><h3>👋 Halo! Tanyakan apa saja tentang film...</h3>"
                    "<p>Percakapan akan nyambung antar giliran 🎬</p></center>",
    )

    # Input row
    with gr.Row(elem_classes="send-row"):
        msg_input = gr.Textbox(
            placeholder="Ketik pertanyaan Anda tentang film...",
            label="Pertanyaan",
            scale=9,
            autofocus=True,
            submit_btn=False,
            max_lines=3,
        )
        send_btn = gr.Button("Kirim 🚀", variant="primary", scale=1, min_width=80)

    with gr.Row():
        reset_btn = gr.Button("🔄 Mulai Percakapan Baru", variant="secondary")

    # Contoh pertanyaan
    gr.Markdown("### 💡 Coba pertanyaan ini:")
    with gr.Row():
        for ex in EXAMPLES[:4]:
            gr.Button(ex, size="sm").click(fn=lambda e=ex: e, outputs=msg_input)
    with gr.Row():
        for ex in EXAMPLES[4:]:
            gr.Button(ex, size="sm").click(fn=lambda e=ex: e, outputs=msg_input)
    gr.Markdown("---")
    gr.Markdown(
        "<center><small>"
        "Ditenagai oleh <b>Groq (LLaMA 3.3 70B)</b> · <b>FAISS</b> · <b>Sentence Transformers</b>"
        "<br>Dataset: IMDB Movies (~10K judul) · Multi-turn RAG · API: GRATIS"
        "</small></center>"
    )

    # Event handlers
    send_btn.click(
        fn=gradio_chat,
        inputs=[msg_input, chatbot_display, session_history],
        outputs=[msg_input, chatbot_display, session_history],
    )
    msg_input.submit(
        fn=gradio_chat,
        inputs=[msg_input, chatbot_display, session_history],
        outputs=[msg_input, chatbot_display, session_history],
    )
    reset_btn.click(
        fn=reset_session,
        outputs=[chatbot_display, session_history],
    )

print("✅ Gradio UI berhasil dibuat.")

/tmp/ipykernel_1434/4126663289.py:32: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_1434/4126663289.py:32: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_1434/4126663289.py:85: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot_display = gr.Chatbot(
/tmp/ipykernel_1434/4126663289.py:85: DeprecationWarning: The 'show_copy_button' parameter will be removed in Gradio 6.0. You will need to use 'buttons=["copy"]' instead.
  chatbot_display = gr.Chatbot(


✅ Gradio UI berhasil dibuat.


In [ ]:
# ─── Launch ────────────────────────────────────────────────────────────────────
demo.launch(
    share=True,          # True → link publik (cocok Google Colab)
    server_name="0.0.0.0",
    server_port=7860,
    show_error=True,
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://50c219a4811597621e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


---

## 📝 Catatan & Tips

### Groq — Model yang Tersedia Gratis

| Model | Kecepatan | Kualitas | Context |
|-------|-----------|----------|---------|
| `llama-3.3-70b-versatile` | ⭐⭐⭐ | ⭐⭐⭐⭐⭐ | 128K |
| `llama-3.1-8b-instant` | ⭐⭐⭐⭐⭐ | ⭐⭐⭐ | 128K |
| `gemma2-9b-it` | ⭐⭐⭐⭐ | ⭐⭐⭐ | 8K |
| `mixtral-8x7b-32768` | ⭐⭐⭐ | ⭐⭐⭐⭐ | 32K |

Ganti model di variabel `LLM_MODEL` di cell konfigurasi.

### Alternatif API Gratis Lain

| Provider | Model Gratis | Cara Ganti |
|----------|-------------|------------|
| **Google Gemini** | `gemini-1.5-flash` | `pip install google-generativeai` |
| **OpenRouter** | `meta-llama/llama-3.1-8b-instruct:free` | kompatibel OpenAI SDK |
| **Cohere** | `command-r` | `pip install cohere` |

### Deploy ke Hugging Face Spaces
1. Ekspor fungsi ke `app.py`
2. Upload: `app.py`, `requirements.txt`, `movie_faiss.index`, `movie_docs.pkl`
3. Set `GROQ_API_KEY` di **Settings → Secrets**
4. Pilih SDK: **Gradio**

```txt
# requirements.txt
groq
sentence-transformers
faiss-cpu
gradio
pandas
numpy
```